In [4]:

# =========================
# IMPROVED ALL-IN-ONE CELL
# =========================

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers

# =========================
# DATASET
# =========================
text = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance."""

# =========================
# TOKENIZATION (LSTM)
# =========================
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)

input_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    input_sequences, maxlen=max_len, padding='pre'
)

X_lstm = input_sequences[:, :-1]
y_lstm = input_sequences[:, -1]
y_lstm = to_categorical(y_lstm, num_classes=total_words)

# =========================
# LSTM MODEL
# =========================
lstm_model = tf.keras.Sequential([
    layers.Embedding(total_words, 64),
    layers.LSTM(150),
    layers.Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam')
lstm_model.fit(X_lstm, y_lstm, epochs=200, verbose=0)

# =========================
# TEMPERATURE SAMPLING
# =========================
def sample_with_temp(preds, temperature=0.7):
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

# =========================
# LSTM GENERATION
# =========================
def generate_lstm(seed_text, next_words=12):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=max_len-1, padding='pre'
        )

        preds = lstm_model.predict(token_list, verbose=0)[0]
        predicted = sample_with_temp(preds, 0.7)

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                if word not in seed_text.split()[-3:]:  # reduce repetition
                    seed_text += " " + word
                break
    return seed_text

print("\n🔥 LSTM Output:")
print(generate_lstm("deep learning", 12))


# =========================
# TRANSFORMER PART
# =========================
vectorizer = layers.TextVectorization(output_mode='int')
vectorizer.adapt([text])

vocab_size = len(vectorizer.get_vocabulary())
sequences = vectorizer([text])[0].numpy()

seq_length = 7   # increased context
X = []
y = []

for i in range(len(sequences) - seq_length):
    X.append(sequences[i:i+seq_length])
    y.append(sequences[i+seq_length])

X = np.array(X)
y = np.array(y)

# Positional Embedding
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embeddings = layers.Embedding(vocab_size, embed_dim)
        self.position_embeddings = layers.Embedding(sequence_length, embed_dim)

    def call(self, inputs):
        positions = tf.range(start=0, limit=tf.shape(inputs)[-1], delta=1)
        return self.token_embeddings(inputs) + self.position_embeddings(positions)

# Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        return self.layernorm2(out1 + self.ffn(out1))

# Transformer Model
embed_dim = 128
num_heads = 4
ff_dim = 128

inputs = layers.Input(shape=(seq_length,))
x = PositionalEmbedding(seq_length, vocab_size, embed_dim)(inputs)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer_model = tf.keras.Model(inputs, outputs)
transformer_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

transformer_model.fit(X, y, epochs=80, verbose=0)

# =========================
# TRANSFORMER GENERATION
# =========================
def generate_transformer(seed, steps=12):
    for _ in range(steps):
        tokenized = vectorizer([seed])[0].numpy()[-seq_length:]
        tokenized = np.pad(tokenized, (seq_length-len(tokenized), 0))

        preds = transformer_model.predict(tokenized.reshape(1, -1), verbose=0)[0]
        next_word_id = sample_with_temp(preds, 0.8)

        next_word = vectorizer.get_vocabulary()[next_word_id]

        if next_word not in seed.split()[-3:]:
            seed += " " + next_word

    return seed

print("\n🚀 Transformer Output:")
print(generate_transformer("machine learning", 12))



🔥 LSTM Output:
deep learning improves sequence modeling performance

🚀 Transformer Output:
machine learning improves sequence generation control lstm sequence models is intelligence deep distributions learning
